In [1]:
# File: homework10

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy import units as u
from astropy import constants as const

In [26]:
df = pd.read_csv("../../course_assignments/homework9/solar_system.csv")

In [27]:
print("Original Shape:", df.shape)

df = df.set_index("Attribute").T
df.index.name = "Planet"
df.reset_index(inplace=True)

print("Transposed Shape:", df.shape) 
print("Columns:", df.columns) 
print(df.head())

Original Shape: (20, 11)
Transposed Shape: (10, 21)
Columns: Index(['Planet', 'Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)',
       'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)',
       'Length of Day (hours)', 'Distance from Sun (10^6 km)',
       'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Orbital Period (days)',
       'Orbital Velocity (km/s)', 'Orbital Inclination (deg)',
       'Orbital Eccentricity', 'Obliquity to Orbit (deg)',
       'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons',
       'Ring System?', 'Global Magnetic Field?'],
      dtype='object', name='Attribute')
Attribute   Planet Mass (10^24kg) Diameter (km) Density (kg/m^3)  \
0          Mercury          0.330          4879             5429   
1            Venus           4.87         12104             5243   
2            Earth           5.97         12756             5514   
3             Moon          0.073          3475             3340   
4             Mars

In [28]:
#Investigate Data Types

In [29]:
print(df.dtypes)

for col in df.columns:
    print(f"Column: {col}")
    print(df[col].apply(type).head(3))
    print("-" * 20)

Attribute
Planet                         object
Mass (10^24kg)                 object
Diameter (km)                  object
Density (kg/m^3)               object
Gravity (m/s^2)                object
Escape Velocity (km/s)         object
Rotation Period (hours)        object
Length of Day (hours)          object
Distance from Sun (10^6 km)    object
Perihelion (10^6 km)           object
Aphelion (10^6 km)             object
Orbital Period (days)          object
Orbital Velocity (km/s)        object
Orbital Inclination (deg)      object
Orbital Eccentricity           object
Obliquity to Orbit (deg)       object
Mean Temperature (C)           object
Surface Pressure (bars)        object
Number of Moons                object
Ring System?                   object
Global Magnetic Field?         object
dtype: object
Column: Planet
0    <class 'str'>
1    <class 'str'>
2    <class 'str'>
Name: Planet, dtype: object
--------------------
Column: Mass (10^24kg)
0    <class 'str'>
1    <class 'st

In [30]:
# Convert Strings to Numbers

In [31]:
cols_to_skip = ["Planet", "Ring System?", "Global Magnetic Field?"]

for col in df.columns:
    if col not in cols_to_skip:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [32]:
print(df.dtypes)

Attribute
Planet                          object
Mass (10^24kg)                 float64
Diameter (km)                    int64
Density (kg/m^3)                 int64
Gravity (m/s^2)                float64
Escape Velocity (km/s)         float64
Rotation Period (hours)        float64
Length of Day (hours)          float64
Distance from Sun (10^6 km)    float64
Perihelion (10^6 km)           float64
Aphelion (10^6 km)             float64
Orbital Period (days)          float64
Orbital Velocity (km/s)        float64
Orbital Inclination (deg)      float64
Orbital Eccentricity           float64
Obliquity to Orbit (deg)       float64
Mean Temperature (C)             int64
Surface Pressure (bars)        float64
Number of Moons                  int64
Ring System?                    object
Global Magnetic Field?          object
dtype: object


In [33]:
# Apply Astropy Units

In [37]:
def attach_units(column_name, unit):
    new_values = [val * unit for val in df[column_name]]
    df[column_name] = new_values
    pass
    
if "Mass (10^24kg)" in df.columns:
    unit = 1e24 * u.kg
    old_col = "Mass (10^24kg)"
    new_col = "Mass (kg)"
    
    new_values = []
    for val in df[old_col]:
        new_values.append(val * unit)
    
    df[old_col] = new_values
    df.rename(columns={old_col: new_col}, inplace=True)

In [38]:
print(df.head(1))

Attribute   Planet   Mass (kg)  Diameter (km)  Density (kg/m^3)  \
0          Mercury  3.3e+23 kg           4879              5429   

Attribute  Gravity (m/s^2)  Escape Velocity (km/s)  Rotation Period (hours)  \
0                      3.7                     4.3                   1407.6   

Attribute  Length of Day (hours)  Distance from Sun (10^6 km)  \
0                         4222.6                         57.9   

Attribute  Perihelion (10^6 km)  ...  Orbital Period (days)  \
0                          46.0  ...                   88.0   

Attribute  Orbital Velocity (km/s)  Orbital Inclination (deg)  \
0                             47.4                        7.0   

Attribute  Orbital Eccentricity  Obliquity to Orbit (deg)  \
0                         0.206                     0.034   

Attribute  Mean Temperature (C)  Surface Pressure (bars)  Number of Moons  \
0                           167                      0.0                0   

Attribute  Ring System? Global Magnetic

In [39]:
# data analysis

In [40]:
if "Aphelion (10^6 km)" in df.columns:
    semi_major = (df["Perihelion (10^6 km)"] + df["Aphelion (10^6 km)"]) / 2 * 1e6 * u.km
    loc_index = df.columns.get_loc("Aphelion (10^6 km)") + 1
    df.insert(loc_index, "Semi-Major Axis (km)", semi_major)
    print(df["Semi-Major Axis (km)"])

0    5.790000e+07
1    1.082000e+08
2    1.496000e+08
3    3.845000e+05
4    2.280000e+08
5    7.785000e+08
6    1.432050e+09
7    2.867050e+09
8    4.515000e+09
9    5.906350e+09
Name: Semi-Major Axis (km), dtype: float64


In [41]:
if "Orbital Period (days)" in df.columns:
    orb_periods_years = [x * u.day for x in df["Orbital Period (days)"]] 
    orb_periods_years = [x.to(u.yr) for x in orb_periods_years]
    df["Orbital Period (days)"] = orb_periods_years
    df.rename(columns={"Orbital Period (days)": "Orbital Period (years)"}, inplace=True)
    print(df.columns)
    print(df["Orbital Period (years)"])
    fav_planet_period = df.loc[2, "Orbital Period (years)"]
    print(f"Earth Orbital Period: {fav_planet_period.value:.4f} years")

Index(['Planet', 'Mass (kg)', 'Diameter (km)', 'Density (kg/m^3)',
       'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)',
       'Length of Day (hours)', 'Distance from Sun (10^6 km)',
       'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Semi-Major Axis (km)',
       'Orbital Period (years)', 'Orbital Velocity (km/s)',
       'Orbital Inclination (deg)', 'Orbital Eccentricity',
       'Obliquity to Orbit (deg)', 'Mean Temperature (C)',
       'Surface Pressure (bars)', 'Number of Moons', 'Ring System?',
       'Global Magnetic Field?'],
      dtype='object', name='Attribute')
0    0.24093086926762491 yr
1     0.6151950718685831 yr
2     0.9998631074606433 yr
3    0.07474332648870637 yr
4     1.8809034907597535 yr
5      11.85763175906913 yr
6     29.423682409308693 yr
7      83.74811772758385 yr
8     163.72347707049965 yr
9     247.93976728268308 yr
Name: Orbital Period (years), dtype: object
Earth Orbital Period: 0.9999 years


In [44]:
df.to_csv("units.csv", index=False)
print("Saved units.csv")

Saved units.csv
